# Election Evaluation

The `ElectionEvaluator` class runs all three evaluation steps automatically in the correct order:
1. Seat allocation (`SeatDistributor`)
2. Constituency count determination (`ConstituencyCountDeterminer`)
3. Constituency assignment (`ConstituencyAssigner`)

This notebook demonstrates the standard path via `evaluate()` as well as each individual step with its configuration options.

For a conceptual overview see [Introduction](../../docs/source/en/einfuehrung.md).

In [ ]:
import pandas as pd
from ipres import (
    Election, ElectionConfig, ConstituenciesConfig,
    ElectionEvaluator, SeatDistributor, ConstituencyCountDeterminer, ConstituencyAssigner,
    SeatDistributionMethod, SuperMajorityMargin, MarginUnit,
    VoteMatrixAnalyzer,
)
from ipres.election_config import (
    ConstituencyRepresentation, QuotaCorrectionStrategy,
)
from ipres.allocation import ConstituencyAllocationMethod

## Setup

All examples in this notebook use the same election: 5 constituencies, 3 parties, non-uniform vote distribution.

In [ ]:
cc = ConstituenciesConfig.from_dataframe(pd.DataFrame({
    'constituency_name': ['C1', 'C2', 'C3', 'C4', 'C5'],
    'constituency_size': [100_000] * 5,
}))
config = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B', 'C'],
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
    seed=42,
)

votes = {
    'C1': {'A': 70, 'B': 20, 'C': 10},
    'C2': {'A': 50, 'B': 35, 'C': 15},
    'C3': {'A': 55, 'B': 25, 'C': 20},
    'C4': {'A': 65, 'B': 25, 'C': 10},
    'C5': {'A': 60, 'B': 30, 'C': 10},
}

election = Election(electionConfig=config)
election.start(votes=votes)

---

## `ElectionEvaluator.evaluate()` — The Simple Path

`evaluate()` runs all three evaluation steps in a single call and returns an `ElectionResult` object.

In [ ]:
evaluator = ElectionEvaluator(seed=42)
result = evaluator.evaluate(election)

In [ ]:
result.get_seat_distribution_table()

In [ ]:
result.get_constituency_summary_table()

In [ ]:
result.get_constituency_assignment_table()

In [ ]:
result.plot_seat_share_pie(min_seats_for_display=1)

---

## Step 1: Seat Allocation — `SeatDistributor`

The `SeatDistributor` distributes parliamentary seats among the parties. Two cases apply:
- **Case 1 (assigned majority)**: The winning party receives the governing majority directly; the remaining seats are distributed proportionally among the other parties.
- **Case 2 (proportional)**: All parties receive their seats proportionally to the election result.

For details see [Seat Allocation](../../docs/source/en/seat_allocation.md).

### `seat_distribution_method`

Three apportionment methods are available for the proportional seat distribution: `SAINTE_LAGUE` (default), `D_HONDT`, and `HARE_NIEMEYER`.

In [ ]:
for method in SeatDistributionMethod:
    seats = SeatDistributor(seat_distribution_method=method).run(election)
    print(f"{method.name:15s} → {seats}")

---

## Step 2: Constituency Count Determination — `ConstituencyCountDeterminer`

Half of a party's allocated parliamentary seats are filled via direct constituency mandates. Therefore a party's constituency count is approximately half its number of seats (`seats // 2`).

For details see [Constituency Count Determination](../../docs/source/en/constituency_count_determination.md).

### `quota_correction_strategy`

For parties with an **odd** seat count, integer division creates a deficit: the sum of the base allocations is smaller than the total number of constituencies. The correction strategy decides which party receives the missing `+1`.

In this example (5 constituencies): A=6, B=3, C=1 seats → base allocation A=3, B=1, C=0 → sum=4, deficit=1. Only B (3 seats) and C (1 seat) have odd seat counts.

In [ ]:
seats = SeatDistributor().run(election)
print("Seats:          ", seats)
print("Base allocation:", {p: s // 2 for p, s in seats.items()})
print(f"Sum of base allocation: {sum(s // 2 for s in seats.values())}  (constituencies: 5 → deficit: 1)")
print()

for strategy in [QuotaCorrectionStrategy.FAVOR_LARGE_PARTIES,
                 QuotaCorrectionStrategy.FAVOR_SMALL_PARTIES]:
    counts = ConstituencyCountDeterminer(quota_correction_strategy=strategy).run(election, seats)
    print(f"{strategy.name}: {counts}")

### `constituency_representation`

This parameter controls which parties receive constituencies at all:
- **`ENTIRE_PARLIAMENT`** *(default)*: All parties receive constituencies proportional to their seats.
- **`GOVERNING_MAJORITY`**: Only the governing party (or parties) receive constituencies. Since all constituencies still need to be represented, the parliament must be larger — `ElectionConfig` computes the required parliament size automatically when this value is set there.

The mode is **not** passed directly to `ConstituencyCountDeterminer` or `ConstituencyAssigner`; instead it is read automatically from `election.electionConfig.constituency_representation`. For the comparison below, two separate `Election` objects with different `ElectionConfig` instances are therefore needed.

For comparison: `ENTIRE_PARLIAMENT` results in a parliament with 10 seats (2 × 5 constituencies), `GOVERNING_MAJORITY` results in 18 seats so that the governing majority alone can cover all 5 constituencies.

In [ ]:
# ENTIRE_PARLIAMENT: existing election (10 seats)
counts_ep = ConstituencyCountDeterminer().run(election, seats)
print(f"ENTIRE_PARLIAMENT (10 seats):  {counts_ep}  → total: {sum(counts_ep.values())}")

# GOVERNING_MAJORITY: new election with matching parliament configuration (18 seats)
config_gm = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B', 'C'],
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
    constituency_representation=ConstituencyRepresentation.GOVERNING_MAJORITY,
    seed=42,
)
election_gm = Election(electionConfig=config_gm)
election_gm.start(votes=votes)
seats_gm = SeatDistributor().run(election_gm)
counts_gm = ConstituencyCountDeterminer().run(election_gm, seats_gm)
print(f"GOVERNING_MAJORITY ({config_gm.parliamentarySeats} seats): {counts_gm}  → total: {sum(counts_gm.values())}")

---

## Step 3: Constituency Assignment — `ConstituencyAssigner`

In the final step, each constituency is assigned to exactly one party for representation. The guiding principle: a constituency should be represented by the party for which it is most important relative to that party's other constituencies.

For details see [Constituency Assignment](../../docs/source/en/constituency_assignment.md).

### Importance Matrix

The importance `w_ij` of constituency `i` for party `j` is calculated as:

```
w_ij = (M − 1) · r_ij / Σ(r_kj  for all k ≠ i)
```

- `w > 1.0` → constituency is above-average important for this party
- `w < 1.0` → below-average important
- `w = 1.0` → average

In [ ]:
vm = election.getLastIteration().vote_matrix.getVotes()
VoteMatrixAnalyzer(vm).show_constituency_importance_matrix()

### `constituency_allocation_method`

Three methods are available to derive an assignment from the importance matrix:
- **`OPTIMAL`** *(default)*: Hungarian algorithm — maximises the total importance score globally.
- **`GREEDY`**: Iteratively assigns the pair with the highest importance value.
- **`STABLE_MATCHING`**: Gale-Shapley algorithm — no blocking pairs remain.

In [ ]:
counts = ConstituencyCountDeterminer().run(election, seats)  # A=3, B=2, C=0
for method in ConstituencyAllocationMethod:
    assignment = ConstituencyAssigner(constituency_allocation_method=method, seed=42).run(election, counts)
    print(f"{method.name:16s} → {assignment}")

### Why GREEDY Can Produce Suboptimal Results

GREEDY selects the (constituency, party) pair with the **globally highest** importance value at each step — without considering the effect on future assignments.

**Constructed importance matrix** (4 constituencies, 3 parties, quotas A = 2, B = 1, C = 1):

|      |  A  |  B  |  C  |
|:----:|:---:|:---:|:---:|
| WK1  |  7  |  5  |  1  |
| WK2  | **5** |  1  | **6** |
| WK3  |  1  |  5  |  4  |
| WK4  |  1  |  1  |  5  |

**GREEDY steps:**
1. WK1 → A (7) — globally highest value ✓
2. **WK2 → C (6)** — GREEDY claims it, even though A also rates WK2 highly (5)
3. WK3 → B (5)
4. WK4 → A (1) — A is left with the weakest constituency

GREEDY total importance: 7 + 6 + 5 + 1 = **19**

**Optimal solution:** WK1 → A (7), WK2 → A (5), WK3 → B (5), WK4 → C (5)  
OPTIMAL total importance: 7 + 5 + 5 + 5 = **22**

**Cause:** GREEDY picks WK2 → C (6) over WK2 → A (5) because 6 > 5. What it misses: C can instead take WK4 (5) — only 1 point less for C. For A, however, WK2 (5) is **4 points** more valuable than WK4 (1). GREEDY sacrifices 4 units for A to gain only 1 unit for C. The Hungarian algorithm identifies this trade-off and finds the globally optimal solution.

In [ ]:
import numpy as np
from ipres.allocation import GreedyAllocationStrategy, OptimalAllocationStrategy

w = pd.DataFrame({
    'A': [7, 5, 1, 1],
    'B': [5, 1, 5, 1],
    'C': [1, 6, 4, 5],
}, index=['WK1', 'WK2', 'WK3', 'WK4'])

print('Importance matrix:')
print(w.to_string())
print()

quotas = {'A': 2, 'B': 1, 'C': 1}
rng = np.random.default_rng(42)

greedy_result  = GreedyAllocationStrategy().allocate(w, quotas, rng)
optimal_result = OptimalAllocationStrategy().allocate(w, quotas, rng)

def total_importance(allocation, matrix):
    return sum(matrix.loc[c, p] for c, p in sorted(allocation.items()))

greedy_total  = total_importance(greedy_result,  w)
optimal_total = total_importance(optimal_result, w)

print('GREEDY:')
for wk, party in sorted(greedy_result.items()):
    print(f'  {wk} → {party}  (importance: {w.loc[wk, party]})')
print(f'  Total importance: {greedy_total}')

print('\nOPTIMAL:')
for wk, party in sorted(optimal_result.items()):
    print(f'  {wk} → {party}  (importance: {w.loc[wk, party]})')
print(f'  Total importance: {optimal_total}')

print(f'\nGreedy loss: {optimal_total - greedy_total} '
      f'({(optimal_total - greedy_total) / optimal_total:.1%} worse)')

---

## Summary

| Parameter | Class | Default |
|---|---|---|
| `seat_distribution_method` | `ElectionEvaluator` / `SeatDistributor` | `SAINTE_LAGUE` |
| `quota_correction_strategy` | `ElectionEvaluator` / `ConstituencyCountDeterminer` | `FAVOR_LARGE_PARTIES` |
| `constituency_allocation_method` | `ElectionEvaluator` / `ConstituencyAssigner` | `OPTIMAL` |

`constituency_representation` is not passed to `ConstituencyCountDeterminer`, `ConstituencyAssigner`, or `ElectionEvaluator`; it is read automatically from `election.electionConfig.constituency_representation`.